# 04 | Treinamento individual

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** treinar no processo do Jupyter. **Entradas:** snapshot, escopo, algoritmo, lote e concorrência. `CAMPAIGN_MODE="NEW"` cria campanha e experimento novos a cada execução da célula de treino. Para retomar, escolha `RESUME` e informe a chave anterior. **Esperado:** prévia antes do fit.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.academic_ui import snapshot_id,metric_controls,model_table
from src.workbench.academic_data import load,prepare
SNAPSHOT_ID=snapshot_id()
CAMPAIGN_MODE=os.getenv('V4_CAMPAIGN_MODE','NEW')
RESUME_REQUEST_KEY=os.getenv('V4_RESUME_REQUEST_KEY')  # Só usado no modo RESUME.
ITEMS=[{'course_code': '2248', 'algorithm': 'ridge'}]
BATCH_SIZE=2
MAX_WORKERS=1
RUN_TRAINING=os.getenv('V4_RUN_TRAINING','true').lower()=='true'
frame,manifest=load(SNAPSHOT_ID)
display(pd.DataFrame([dict(**item,train_rows=len(prepare(frame,item['course_code'])[0]),evaluation_rows=len(prepare(frame,item['course_code'])[1])) for item in ITEMS]))
display({'recursos':'container Jupyter: limite3GiB; concorrência permitida1..2','targets':'regular,dependency,elective','avaliação':'holdout do último ciclo histórico; métricas sintéticas'})


**Métricas:** selecione os checkboxes antes de executar a próxima célula. A seleção altera de fato o cálculo, o hash da configuração e o tracking. **Esperado:** seleção não vazia e compatível.

In [ ]:

METRIC_PROFILE='forecast_extended'
metric_boxes=metric_controls(METRIC_PROFILE)


**Executar:** fit, predict, métricas, persistência proxy e Registry. **Resultado:** tabelas e links; nenhuma promoção automática.

In [ ]:

from src.workbench.academic_campaign import execute
from uuid import uuid4
from IPython.display import Markdown
selected=[name for name,box in metric_boxes.items() if box.value]
if not selected:
    raise ValueError('Selecione pelo menos uma métrica.')
if RUN_TRAINING:
    if CAMPAIGN_MODE not in ('NEW','RESUME'):
        raise ValueError('CAMPAIGN_MODE deve ser NEW ou RESUME.')
    if CAMPAIGN_MODE=='RESUME' and not RESUME_REQUEST_KEY:
        raise ValueError('Informe RESUME_REQUEST_KEY da campanha anterior; não será criada outra silenciosamente.')
    REQUEST_KEY='v4-notebook-'+str(uuid4()) if CAMPAIGN_MODE=='NEW' else RESUME_REQUEST_KEY
    save_context(campaign_request_key=REQUEST_KEY,campaign_id=None,proposal_id=None)
    display({'modo':CAMPAIGN_MODE,'request_key':REQUEST_KEY,'retomada':'Guarde esta chave para RESUME em caso de interrupção.'})
    result=execute(SNAPSHOT_ID,REQUEST_KEY,items=ITEMS,batch_size=BATCH_SIZE,max_workers=MAX_WORKERS,metric_profile=METRIC_PROFILE,selected_metrics=selected,resume_only=CAMPAIGN_MODE=='RESUME')
    if result['status']!='COMPLETED':
        raise RuntimeError('Campanha parcial; consulte workbench_models e eventos por campaign_id.')
    save_context(campaign_id=result['campaign_id'],snapshot_id=SNAPSHOT_ID,proposal_id=None)
    ui=os.environ['MLFLOW_UI_BASE_URL'].rstrip('/')
    display(Markdown(f"[Abrir experimento {result['experiment_id']}]({ui}/#/experiments/{result['experiment_id']}) · [Abrir campanha]({ui}/#/experiments/{result['experiment_id']}/runs/{result['run_id']})"))
    display(pd.DataFrame([{k:result[k] for k in ('campaign_id','requested','completed','failed','quality_eligible')}]))
    display(model_table(result['models']))
    display(pd.DataFrame(result['aggregation']))
    pd.Series({m['course_code']:m['metrics'].get('regular.mae') for m in result['models']}).plot.bar(title='MAE regular — avaliação sintética',ylabel='estudantes')
else:
    display('NOT_EXECUTED: treinamento desabilitado explicitamente.')


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.